# Test the AgentCore Gateway

Call tools through the Gateway using Cognito M2M credentials. The Gateway adds JWT auth, audit logging, and guardrails — capabilities that raw Lambda invocation doesn't have.

In [ ]:
%pip install "boto3==1.42.87" "httpx==0.27.0" --quiet
import boto3
assert tuple(int(x) for x in boto3.__version__.split(".")[:3]) >= (1, 42, 87), f"boto3 too old: {boto3.__version__} (need >=1.42.87)"
print(f"boto3 {boto3.__version__} OK")

## Step 1: Get M2M Credentials

In [ ]:
import boto3, json, httpx

region = boto3.session.Session().region_name or "us-west-2"

cfn = boto3.client("cloudformation", region_name=region)
exports = {}
for page in cfn.get_paginator("list_exports").paginate():
    for e in page["Exports"]:
        exports[e["Name"]] = e["Value"]

GATEWAY_URL = exports["ac-GatewayUrl"]
TOKEN_URL = exports["workshop-CognitoTokenUrl"]
M2M_SECRET_ARN = exports["workshop-CognitoM2MClientSecretArn"]

# Reset any group-based access policy from prior runs of notebook 07.
# The M2M access token issued in Step 2 has no cognito:groups claim, so any
# non-empty policy would silently deny every tools/call in Steps 4-6 below.
# Notebook 07 re-applies the policy at the end of its own flow.
INTERCEPTOR_FN = "ac-gateway-request-interceptor"
lambda_client = boto3.client("lambda", region_name=region)
_cfg = lambda_client.get_function_configuration(FunctionName=INTERCEPTOR_FN)
_env = _cfg.get("Environment", {}).get("Variables", {}) or {}
if _env.get("TOOL_ACCESS_POLICY"):
    _env["TOOL_ACCESS_POLICY"] = ""
    lambda_client.update_function_configuration(
        FunctionName=INTERCEPTOR_FN,
        Environment={"Variables": _env},
    )
    waiter = lambda_client.get_waiter("function_updated_v2")
    waiter.wait(FunctionName=INTERCEPTOR_FN)
    print(f"Cleared TOOL_ACCESS_POLICY on {INTERCEPTOR_FN} (will be re-set by notebook 07)")

# Get M2M credentials
sm = boto3.client("secretsmanager", region_name=region)
secret = json.loads(sm.get_secret_value(SecretId=M2M_SECRET_ARN)["SecretString"])
CLIENT_ID = secret["client_id"]
CLIENT_SECRET = secret["client_secret"]

print(f"Gateway URL: {GATEWAY_URL}")
print(f"Token URL:   {TOKEN_URL}")
print(f"Client ID:   {CLIENT_ID}")

## Step 2: Get Cognito M2M Token

In [ ]:
import base64

auth_header = base64.b64encode(f"{CLIENT_ID}:{CLIENT_SECRET}".encode()).decode()
token_resp = httpx.post(
    TOKEN_URL,
    headers={"Authorization": f"Basic {auth_header}", "Content-Type": "application/x-www-form-urlencoded"},
    data={"grant_type": "client_credentials", "scope": "mcp-servers-unrestricted/read mcp-servers-unrestricted/execute"},
    timeout=15,
)
token_resp.raise_for_status()
ACCESS_TOKEN = token_resp.json()["access_token"]
print(f"Token obtained (length: {len(ACCESS_TOKEN)})")
HEADERS = {"Authorization": f"Bearer {ACCESS_TOKEN}", "Content-Type": "application/json"}

## Step 2b: WorkloadIdentity Token Flow (Platform-Native Auth)

The manual Cognito flow above works, but it requires the agent to know the `client_id` and `client_secret`. In a production platform, agents should NOT have access to raw credentials.

AgentCore provides a better pattern:
1. **OAuth2CredentialProvider** — stores the Cognito client credentials securely in AgentCore
2. **WorkloadIdentity** — the agent proves its identity, then exchanges it for an OAuth2 token

The agent never sees the client secret. AgentCore manages the credential lifecycle.

```
Agent  ──(1) get_workload_access_token()──>  AgentCore (proves identity)
Agent  ──(2) get_resource_oauth2_token()──>  AgentCore (exchanges for Gateway token)
Agent  ──(3) Bearer token──>  Gateway (calls tools)
```

In [ ]:
# Step 2b.1: Create (or refresh) an OAuth2CredentialProvider.
# This stores the Cognito M2M credentials inside AgentCore so agents never see the raw secret.
#
# Why this block is longer than a simple "create if not exists":
#   OAuth2CredentialProvider entries persist across CloudFormation redeploys.
#   If an earlier deploy created this provider pointing at a *previous* Cognito pool,
#   the provider still exists but its discoveryUrl now points at a pool that no longer resolves.
#   `get_resource_oauth2_token` would then fail with "Invalid Discovery URL".
#   So we check the stored discoveryUrl and replace the provider if it doesn't match this pool.
from botocore.exceptions import ClientError

cp_client = boto3.client("bedrock-agentcore-control", region_name=region)

PROVIDER_NAME = "workshop-gateway-oauth"
POOL_ID = exports["workshop-CognitoUserPoolId"]
EXPECTED_DISCOVERY = f"https://cognito-idp.{region}.amazonaws.com/{POOL_ID}/.well-known/openid-configuration"


def _find_provider(name):
    """Return the provider summary if present, else None. Uses the actual API response key."""
    paginator = cp_client.get_paginator("list_oauth2_credential_providers")
    for page in paginator.paginate():
        for p in page.get("credentialProviders", []):
            if p.get("name") == name:
                return p
    return None


def _create_provider():
    cp_client.create_oauth2_credential_provider(
        name=PROVIDER_NAME,
        credentialProviderVendor="CustomOauth2",
        oauth2ProviderConfigInput={
            "customOauth2ProviderConfig": {
                "oauthDiscovery": {"discoveryUrl": EXPECTED_DISCOVERY},
                "clientId": CLIENT_ID,
                "clientSecret": CLIENT_SECRET,
            }
        },
    )


existing = _find_provider(PROVIDER_NAME)
if existing:
    # Provider found — verify its discovery URL matches THIS deploy's Cognito pool.
    full = cp_client.get_oauth2_credential_provider(name=PROVIDER_NAME)
    stored_discovery = (
        full.get("oauth2ProviderConfigOutput", {})
            .get("customOauth2ProviderConfig", {})
            .get("oauthDiscovery", {})
            .get("discoveryUrl")
    )
    if stored_discovery == EXPECTED_DISCOVERY:
        print(f"CredentialProvider up-to-date: {PROVIDER_NAME}")
    else:
        print(f"CredentialProvider points at stale pool ({stored_discovery!r}); recreating...")
        cp_client.delete_oauth2_credential_provider(name=PROVIDER_NAME)
        _create_provider()
        print(f"Recreated CredentialProvider: {PROVIDER_NAME}")
else:
    try:
        _create_provider()
        print(f"Created CredentialProvider: {PROVIDER_NAME}")
    except ClientError as e:
        code = e.response.get("Error", {}).get("Code", "")
        msg = e.response.get("Error", {}).get("Message", "")
        if code in ("ValidationException", "ConflictException") and "already exists" in msg:
            # Race: someone else created it between list and create — trust it.
            print(f"CredentialProvider already exists: {PROVIDER_NAME}")
        else:
            raise

print(f"\nAgents can now get Gateway tokens without knowing the client secret.")

In [ ]:
# Step 2b.2: Get a Gateway token via WorkloadIdentity (how an agent authenticates)
# This is the data-plane flow — no raw credentials needed
dp_client = boto3.client("bedrock-agentcore", region_name=region)

WORKLOAD_NAME = exports.get("ac-WorkloadIdentityName", "ac-agent-identity")

# Step 1: Prove workload identity
workload_resp = dp_client.get_workload_access_token(
    workloadName=WORKLOAD_NAME,
)
workload_token = workload_resp["workloadAccessToken"]
print(f"1. WorkloadIdentity token acquired (length: {len(workload_token)})")

# Step 2: Exchange for OAuth2 token via CredentialProvider
oauth_resp = dp_client.get_resource_oauth2_token(
    workloadIdentityToken=workload_token,
    resourceCredentialProviderName=PROVIDER_NAME,
    oauth2Flow="M2M",
    scopes=["mcp-servers-unrestricted/read", "mcp-servers-unrestricted/execute"],
)
WORKLOAD_TOKEN = oauth_resp["accessToken"]
print(f"2. Gateway OAuth2 token acquired (length: {len(WORKLOAD_TOKEN)})")
print(f"\nThis token works identically to the manual Cognito token.")
print(f"The agent never saw the client_id or client_secret.")

In [ ]:
# Step 2b.3: Verify the WorkloadIdentity token works with the Gateway
WI_HEADERS = {"Authorization": f"Bearer {WORKLOAD_TOKEN}", "Content-Type": "application/json"}

resp = httpx.post(
    GATEWAY_URL, headers=WI_HEADERS,
    json={"jsonrpc": "2.0", "method": "tools/list", "id": 1}, timeout=30,
)
resp.raise_for_status()
wi_tools = resp.json().get("result", {}).get("tools", [])
print(f"WorkloadIdentity token: Gateway reports {len(wi_tools)} tool(s)")
print(f"\nWorkloadIdentity is the production authentication pattern — no client secret in agent code.")
print(f"(Step 3 below lists the tools explicitly via the manual-token path for comparison.)")

## Step 3: List Tools via Gateway

In [ ]:
resp = httpx.post(
    GATEWAY_URL,
    headers=HEADERS,
    json={"jsonrpc": "2.0", "method": "tools/list", "id": 1},
    timeout=30,
)
resp.raise_for_status()
tools = resp.json().get("result", {}).get("tools", [])
print(f"Gateway reports {len(tools)} tool(s):")
print()

# Build a lookup of short name -> gateway-prefixed name
TOOL_MAP = {}
for t in tools:
    full_name = t["name"]
    # Gateway prefixes tools as: {target-name}___{tool-name}
    short_name = full_name.split("___")[-1] if "___" in full_name else full_name
    TOOL_MAP[short_name] = full_name
    print(f"  {full_name:55s} {t.get('description', '')[:50]}")

print(f"\nTool name mapping (short -> gateway):")
for short, full in TOOL_MAP.items():
    print(f"  {short:30s} -> {full}")

## Step 4: Call search_flights

In [ ]:
# Use the gateway-prefixed tool name discovered from tools/list
tool_name = TOOL_MAP.get("search_flights", "search_flights")
print(f"Calling: {tool_name}")

resp = httpx.post(
    GATEWAY_URL,
    headers=HEADERS,
    json={
        "jsonrpc": "2.0", "method": "tools/call", "id": 2,
        "params": {"name": tool_name, "arguments": {"origin": "SFO", "destination": "TYO", "date": "2026-09-15"}}
    },
    timeout=30,
)
resp.raise_for_status()
result = resp.json()
if "error" in result:
    raise RuntimeError(f"Gateway returned JSON-RPC error: {result['error']}")
content = result.get("result", {}).get("content", [])
if not content:
    raise RuntimeError(f"Gateway returned empty content. Full response: {json.dumps(result, indent=2)}")
for item in content:
    if item.get("type") == "text":
        print(item["text"])

## Step 5: Call search_hotels

In [ ]:
tool_name = TOOL_MAP.get("search_hotels", "search_hotels")
print(f"Calling: {tool_name}")

resp = httpx.post(
    GATEWAY_URL,
    headers=HEADERS,
    json={
        "jsonrpc": "2.0", "method": "tools/call", "id": 3,
        "params": {"name": tool_name, "arguments": {"city": "Tokyo", "check_in": "2026-09-15", "check_out": "2026-09-18"}}
    },
    timeout=30,
)
resp.raise_for_status()
result = resp.json()
if "error" in result:
    raise RuntimeError(f"Gateway returned JSON-RPC error: {result['error']}")
content = result.get("result", {}).get("content", [])
if not content:
    raise RuntimeError(f"Gateway returned empty content. Full response: {json.dumps(result, indent=2)}")
for item in content:
    if item.get("type") == "text":
        print(item["text"])

## Step 6: Call search-knowledge-base

In [ ]:
tool_name = TOOL_MAP.get("search-knowledge-base", "search-knowledge-base")
print(f"Calling: {tool_name}")

resp = httpx.post(
    GATEWAY_URL,
    headers=HEADERS,
    json={
        "jsonrpc": "2.0", "method": "tools/call", "id": 4,
        "params": {"name": tool_name, "arguments": {"query": "return policy", "max_results": 3}}
    },
    timeout=30,
)
resp.raise_for_status()
result = resp.json()
if "error" in result:
    raise RuntimeError(f"Gateway returned JSON-RPC error: {result['error']}")
content = result.get("result", {}).get("content", [])
if not content:
    raise RuntimeError(f"Gateway returned empty content. Full response: {json.dumps(result, indent=2)}")
for item in content:
    if item.get("type") == "text":
        print(item["text"])

## Step 7: Verify Audit Logging

The request interceptor logs every tool call to CloudWatch.

In [ ]:
logs = boto3.client("logs", region_name=region)
LOG_GROUP = "/aws/lambda/ac-gateway-request-interceptor"

try:
    streams = logs.describe_log_streams(
        logGroupName=LOG_GROUP,
        orderBy="LastEventTime",
        descending=True,
        limit=1,
    )["logStreams"]

    if streams:
        events = logs.get_log_events(
            logGroupName=LOG_GROUP,
            logStreamName=streams[0]["logStreamName"],
            limit=5,
        )["events"]
        print(f"Recent audit log entries ({LOG_GROUP}):")
        for evt in events:
            print(f"  {evt['message'].strip()[:120]}")
    else:
        print(f"No log streams in {LOG_GROUP} yet — the interceptor has not been invoked recently.")
except logs.exceptions.ResourceNotFoundException:
    print(f"Log group {LOG_GROUP} does not exist. The interceptor Lambda may not be wired up — "
          f"verify ac-gateway-request-interceptor exists in the workshop-agentcore-stack.")

## Summary

| Action | Method | Result |
|---|---|---|
| Manual Cognito token | OAuth2 client_credentials grant | Access token via raw client_id/secret |
| WorkloadIdentity token | `get_workload_access_token` + `get_resource_oauth2_token` | Access token without raw credentials |
| List all tools | `tools/list` | 7+ tools across 3 targets |
| Search flights | `tools/call search_flights` | SFO->TYO flights |
| Search hotels | `tools/call search_hotels` | Tokyo hotels |
| Search knowledge base | `tools/call search-knowledge-base` | Return policy docs |
| Audit logging | CloudWatch | Every call logged with actor, tool, timestamp |

**Key insight:** The WorkloadIdentity flow is how Module 4 agents should authenticate. The agent proves its identity to AgentCore, which then securely fetches the OAuth2 token on its behalf. No client secrets in agent code.

**Next:** Notebook 07 adds Bedrock Guardrails and group-based access control.